# w17_gpu1_factorial_lowdisp_and_jrdb_conformal
Inputs: `shortpose-wave1-private-inputs` | Accelerator: `GPU T4 x2`


In [ ]:
import hashlib, os, subprocess, sys, tempfile, zipfile
from pathlib import Path
import torch

WORK, MOUNT = Path('/kaggle/working'), Path('/kaggle/input')
assert WORK.is_dir() and torch.cuda.is_available(), 'Kaggle GPU required'
n_gpu = torch.cuda.device_count()
REV = 'dda649afa258c4e35aaa8c23a70bc7d0e934d6eb'
with tempfile.TemporaryDirectory(prefix='w17_g1_') as ts:
    tmp = Path(ts)
    repo, meta_dir = tmp / 'repo', tmp / 'pinns_meta'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    base_env = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1'}
    joint_bin = next(MOUNT.rglob('joint_actions_all.bin'))
    heldout_bin = next(MOUNT.rglob('shortpose_heldout.bin'))
    map_bin = next(MOUNT.rglob('pinns_source_label_causal_maps.bin'))
    pose_npz = tmp / 'pose.npz'
    with zipfile.ZipFile(heldout_bin) as zf:
        pose_npz.write_bytes(zf.read('pose.npz'))
    subprocess.run([sys.executable, '-m', 'scripts.stage_pinns_metadata', '--output-dir', str(meta_dir)], cwd=repo, env=base_env, check=True)
    d0 = WORK / 'wave17_rc1_2x2_factorial_and_lowdisp_regime.zip'
    d1 = WORK / 'wave17_rc3_clean_chain_and_jrdb_conformal.zip'
    e0 = {**base_env, 'CUDA_VISIBLE_DEVICES': '0'}
    e1 = {**base_env, 'CUDA_VISIBLE_DEVICES': '1' if n_gpu > 1 else '0'}
    p0 = subprocess.Popen([sys.executable, '-u', '-m', 'scripts.run_wave17_rc1_2x2_factorial_and_lowdisp_regime', '--repo-root', str(repo), '--meta-dir', str(meta_dir), '--pose-npz', str(pose_npz), '--output-zip', str(d0)], cwd=repo, env=e0)
    p1 = subprocess.Popen([sys.executable, '-u', '-m', 'scripts.run_wave17_rc3_clean_chain_and_jrdb_conformal', '--repo-root', str(repo), '--meta-dir', str(meta_dir), '--pose-npz', str(pose_npz), '--joint-zip', str(joint_bin), '--causal-map-bin', str(map_bin), '--output-zip', str(d1)], cwd=repo, env=e1)
    rc0, rc1 = p0.wait(), p1.wait()
    assert rc0 == 0 and rc1 == 0, f'Subprocess failed: rc0={rc0}, rc1={rc1}'
    for d in (d0, d1):
        print('DONE:', d, d.stat().st_size, hashlib.sha256(d.read_bytes()).hexdigest(), flush=True)
